In [23]:
import tkinter as tk
from tkinter import ttk
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import time
from collections import deque
from datetime import datetime

class SumGraphApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Suma de Teclas Presionadas con Gráfico")
        self.root.geometry("900x650")
        
        # Inicialización de variables de control
        self.current_sum = 0
        self.start_time = time.time()
        self.last_sample_time = self.start_time
        
        # Estado de las teclas 1 a 5
        self.pressed_keys = {1: False, 2: False, 3: False, 4: False, 5: False}
        
        # Colas para almacenar datos de los últimos 30 segundos
        self.time_data = deque()
        self.sum_data = deque()
        
        # Configuración de interfaz y gráfico
        self.setup_gui()
        self.setup_plot()
        self.bind_keys()
        
        # Punto inicial
        self.time_data.append(self.start_time)
        self.sum_data.append(0)
        
        # Iniciar procesos de muestreo y actualización visual
        self.start_sampling()
        self.update_graph()
        
    def setup_gui(self):
        # Contenedor principal
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(1, weight=1)
        main_frame.columnconfigure(0, weight=1)
        
        # Etiquetas de suma y valor máximo
        sum_frame = ttk.Frame(main_frame)
        sum_frame.grid(row=0, column=0, columnspan=3, pady=10)
        
        self.sum_label = ttk.Label(sum_frame, text="Suma Actual: 0", 
                                   font=("Arial", 28, "bold"))
        self.sum_label.pack()
        
        self.max_label = ttk.Label(sum_frame, text="Máximo posible: 15", 
                                   font=("Arial", 10))
        self.max_label.pack()
        
        # Indicadores visuales para cada tecla
        self.key_frames = {}
        keys_frame = ttk.Frame(main_frame)
        keys_frame.grid(row=1, column=0, columnspan=3, pady=10)
        
        for i in range(1, 6):
            frame = ttk.LabelFrame(keys_frame, text=f"Tecla {i}", width=100, height=100)
            frame.grid(row=0, column=i-1, padx=10, pady=5)
            frame.grid_propagate(False)
            
            value_label = ttk.Label(frame, text=f"Valor: {i}", font=("Arial", 14, "bold"))
            value_label.pack(expand=True, pady=(15, 5))
            
            status = ttk.Label(frame, text="● Libre", foreground="gray")
            status.pack()
            
            self.key_frames[i] = frame
            self.key_frames[f"status_{i}"] = status
        
        # Botones de control
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=2, column=0, columnspan=3, pady=10)
        
        self.reset_btn = ttk.Button(button_frame, text="Reiniciar Contador", 
                                   command=self.reset_counter, width=20)
        self.reset_btn.grid(row=0, column=0, padx=5)
        
        self.quit_btn = ttk.Button(button_frame, text="Fin de Programa", 
                                   command=self.quit_program, width=20)
        self.quit_btn.grid(row=0, column=1, padx=5)
        
        # Textos informativos
        self.info_label = ttk.Label(main_frame, 
                                   text="Presione y suelte las teclas 1, 2, 3, 4, 5",
                                   font=("Arial", 10))
        self.info_label.grid(row=3, column=0, columnspan=3, pady=10)
        
        self.current_keys_label = ttk.Label(main_frame, 
                                            text="Teclas presionadas actualmente: Ninguna",
                                            font=("Arial", 10, "italic"))
        self.current_keys_label.grid(row=4, column=0, columnspan=3, pady=5)
        
        # Cuadro de historial de eventos
        history_frame = ttk.LabelFrame(main_frame, text="Historial de Eventos")
        history_frame.grid(row=5, column=0, columnspan=3, pady=10, sticky=(tk.W, tk.E))
        
        self.history_text = tk.Text(history_frame, height=8, width=80)
        scrollbar = ttk.Scrollbar(history_frame, orient="vertical", command=self.history_text.yview)
        self.history_text.configure(yscrollcommand=scrollbar.set)
        self.history_text.pack(side="left", fill="both", expand=True, padx=5, pady=5)
        scrollbar.pack(side="right", fill="y")
        
    def setup_plot(self):
        # Crear figura de Matplotlib e integrarla en Tkinter
        self.fig, self.ax = plt.subplots(figsize=(8, 4))
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.root)
        self.canvas.get_tk_widget().grid(row=1, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), padx=10, pady=5)
        
        self.ax.set_xlabel("Tiempo (segundos desde inicio)")
        self.ax.set_ylabel("Suma de Teclas Presionadas")
        self.ax.set_title("Suma de Teclas vs Tiempo (últimos 30s)")
        self.ax.grid(True, alpha=0.3)
        self.ax.set_ylim(0, 16)
        self.ax.set_yticks(range(0, 17, 1))
        
    def bind_keys(self):
        # Vincular eventos de presionar y soltar para teclas 1-5
        for i in range(1, 6):
            self.root.bind(f"<KeyPress-{i}>", lambda event, val=i: self.key_press(val))
            self.root.bind(f"<KeyRelease-{i}>", lambda event, val=i: self.key_release(val))
    
    def sample_current_value(self):
        # Guardar el valor actual si cambió o si pasó tiempo suficiente
        current_time = time.time()
        
        if not self.time_data or self.sum_data[-1] != self.current_sum:
            self.time_data.append(current_time)
            self.sum_data.append(self.current_sum)
        elif current_time - self.last_sample_time >= 0.1:
            self.time_data.append(current_time)
            self.sum_data.append(self.current_sum)
            self.last_sample_time = current_time
        
    def start_sampling(self):
        # Bucle de muestreo cada 50ms
        def sample():
            self.sample_current_value()
            self.root.after(50, sample)
        sample()
    
    def update_current_keys_display(self):
        # Mostrar qué teclas están activas en texto
        pressed = [str(k) for k, v in self.pressed_keys.items() if v]
        if pressed:
            self.current_keys_label.config(text=f"Teclas presionadas actualmente: {', '.join(pressed)}")
        else:
            self.current_keys_label.config(text="Teclas presionadas actualmente: Ninguna")
    
    def update_sum_display(self):
        # Recalcular suma total y actualizar interfaz
        self.current_sum = sum(key for key, pressed in self.pressed_keys.items() if pressed)
        self.sum_label.config(text=f"Suma Actual: {self.current_sum}")
        
        # Cambiar color según la suma
        if self.current_sum == 15:
            self.sum_label.configure(foreground="green")
        elif self.current_sum > 0:
            self.sum_label.configure(foreground="blue")
        else:
            self.sum_label.configure(foreground="black")
        
        self.update_current_keys_display()
        
        # Registrar evento en el historial
        pressed_keys = [str(k) for k, v in self.pressed_keys.items() if v]
        pressed_str = ", ".join(pressed_keys) if pressed_keys else "ninguna"
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.history_text.insert(tk.END, f"{timestamp}: Teclas: {pressed_str} → Suma: {self.current_sum}\n")
        self.history_text.see(tk.END)
        
    def key_press(self, value):
        # Lógica al presionar tecla
        if not self.pressed_keys[value]:
            self.pressed_keys[value] = True
            self.key_frames[value].configure(style="Pressed.TLabelframe")
            self.key_frames[f"status_{value}"].configure(text="● Presionada", foreground="green")
            self.update_sum_display()
    
    def key_release(self, value):
        # Lógica al soltar tecla
        if self.pressed_keys[value]:
            self.pressed_keys[value] = False
            self.key_frames[value].configure(style="TLabelframe")
            self.key_frames[f"status_{value}"].configure(text="● Libre", foreground="gray")
            self.update_sum_display()
    
    def reset_counter(self):
        # Resetear todos los estados a falso
        current_time = time.time()
        for i in range(1, 6):
            if self.pressed_keys[i]:
                self.pressed_keys[i] = False
                self.key_frames[i].configure(style="TLabelframe")
                self.key_frames[f"status_{i}"].configure(text="● Libre", foreground="gray")
        
        self.current_sum = 0
        self.sum_label.config(text=f"Suma Actual: 0")
        self.update_current_keys_display()
        
        self.time_data.append(current_time)
        self.sum_data.append(0)
        
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.history_text.insert(tk.END, f"{timestamp}: 🔄 REINICIO\n")
        self.history_text.see(tk.END)
        
    def quit_program(self):
        # Cerrar la aplicación
        self.root.quit()
        self.root.destroy()
        
    def update_graph(self):
        # Actualizar visualización del gráfico
        current_time = time.time()
        time_threshold = current_time - 30
        
        # Eliminar datos antiguos fuera del rango de 30s
        while self.time_data and self.time_data[0] < time_threshold:
            self.time_data.popleft()
            self.sum_data.popleft()
        
        if self.time_data:
            times = [t - self.start_time for t in self.time_data]
            self.ax.clear()
            
            # Dibujar línea azul
            self.ax.plot(times, self.sum_data, 'b-', linewidth=2)
            
            # Resaltar puntos de cambio en rojo
            if len(times) > 1:
                changes = [i for i in range(1, len(self.sum_data)) if self.sum_data[i] != self.sum_data[i-1]]
                for i in changes:
                    self.ax.plot(times[i], self.sum_data[i], 'ro', markersize=6)
            
            # Re-configurar estética tras limpiar
            self.ax.set_xlabel("Segundos")
            self.ax.set_ylabel("Suma")
            self.ax.set_title("Historial de Suma (30s)")
            self.ax.grid(True, alpha=0.3)
            self.ax.set_ylim(0, 16)
            self.ax.set_yticks(range(0, 17, 1))
            
            # Ajustar ventana de tiempo
            max_t = current_time - self.start_time
            self.ax.set_xlim(max(max_t - 30, 0), max_t)
            
            # Línea horizontal del valor actual
            self.ax.axhline(y=self.current_sum, color='r', linestyle='--', alpha=0.5, label=f'Actual: {self.current_sum}')
            self.ax.legend(loc='upper left')
            
            self.canvas.draw()
        
        # Re-programar actualización cada 100ms
        self.root.after(100, self.update_graph)

def configure_styles():
    # Estilos para cuando la tecla está presionada
    style = ttk.Style()
    style.configure("Pressed.TLabelframe", background="#90EE90", relief="sunken")
    style.configure("Pressed.TLabelframe.Label", foreground="darkgreen", font=("Arial", 10, "bold"))

def main():
    root = tk.Tk()
    configure_styles()
    app = SumGraphApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()

Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not NoneType
Error: 'in <string>' requires string as left operand, not None

KeyboardInterrupt: 